# Ideal-spectrum RV recovery (single-line Gaussian centroid)

Goals:

1. **Non-approximate Doppler paint** — densify the rest spectrum until Lanczos
   resampling onto instrument WAVE matches an analytical Doppler shift to
   machine precision (Gaussian line), and converges for BT-Settl.
2. Use the **same spectrum interpolator** as the science `ccf_line_mask` path:
   Numba **Lanczos-a** (`doppler_paint_lanczos` / `_poly_interp_sorted_1d`,
   `a` from `mask_ccf_pixel_interp_order`, default 3 = Lanczos-3 windowed-sinc
   in pixel-index space).
3. Final RV = **Gaussian CCF centroid only** (parabolic-vertex refine removed).
4. Debug with **one mask line**.


In [1]:
from pathlib import Path

import numpy as np
import astropy.units as u

import hispecdrp
from hispecdrp import datamodels
from hispecdrp.datamodels import hispec_config

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

OUT = Path("test_data/ideal_spectrum_rv_recovery")
OUT.mkdir(parents=True, exist_ok=True)
print("output:", OUT.resolve())


hispecdrp 0.0.post2+waveinject.local @ /Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/__init__.py
output: /Users/ckilpatrick/Library/CloudStorage/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/simulations/notebooks/test_data/ideal_spectrum_rv_recovery


In [2]:
from hispecdrp.primitives.utils.ccf import doppler_paint_lanczos
from hispecdrp.simulations.simulate_rate_map import SimulateRateMap
from hispecdrp.utils.line_mask_utils import MaskLines
from hispecdrp.utils.physical_constants import C_M_S

SPECTROGRAPH = "BSPEC"
FIBER = 1
ORDER = 130  # single echelle order for one-line debug
BULK_RV_KM_S = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0]

# Line width and CCF window must be consistent: Gaussian LM centroid needs the
# full core+wings. A half-window of only ~0.5σ truncates the peak and drives
# km/s-level fit bias even when discrete argmax is near the truth.
sig_v_m_s = 2000.0  # Gaussian line σ in velocity (~few BSPEC pixels)
V_STEP_M_S = 1.0
V_HALF_WINDOW_M_S = 5.0 * sig_v_m_s  # ±5σ

# Paint / recovery thresholds
BSPEC_INTERP_MAX_ABS = 1e-5  # Lanczos-3 vs continuous on BSPEC samples
RV_BIAS_MAX_M_S = 5e-2  # 5 cm/s ceiling for single-line Gaussian centroid
MASK_CCF_PIXEL_INTERP_ORDER = 3  # same default as science ccf_line_mask

ws_all = SimulateRateMap().get_wavelength_solution(spectrograph=SPECTROGRAPH)[FIBER]
wave = np.asarray(ws_all[ORDER]["wavelength"], dtype=np.float64)
assert np.all(np.diff(wave) > 0.0)
lam0 = float(wave[wave.size // 2])
sig_lam = lam0 * sig_v_m_s / float(C_M_S)

print(f"order={ORDER}  n_pix={wave.size}  λ0={lam0:.6f} µm  σ_λ={sig_lam:.3e} µm")
print(f"CCF: v_step={V_STEP_M_S} m/s  half_window={V_HALF_WINDOW_M_S} m/s")
print(f"interpolator: Lanczos-a={MASK_CCF_PIXEL_INTERP_ORDER} (same as mask CCF)")


order=130  n_pix=4096  λ0=1.123321 µm  σ_λ=3.747e-05 µm
CCF: v_step=1.0 m/s  half_window=5000.0 m/s
interpolator: Lanczos-a=3 (same as mask CCF)


## 1. Lanczos on the BSPEC grid vs continuous spectrum

Sample an analytical Gaussian onto instrument WAVE, then evaluate
`_poly_interp_sorted_1d` / `doppler_paint_lanczos` at half-pixel wavelengths.
Compare to the continuous profile (the CCF use-case: arbitrary λ between pixels).


In [3]:
from hispecdrp.primitives.utils.ccf.ccf_utils import _poly_interp_sorted_1d

def gaussian_rest(w, lam_c, sig):
    return np.exp(-0.5 * ((w - lam_c) / sig) ** 2)


# Well-resolved line on BSPEC (~tens of pixels across σ).
sig_v_bspec = 20000.0  # m/s
sig_bspec = lam0 * sig_v_bspec / float(C_M_S)
dlam = float(np.median(np.diff(wave)))
print(f"BSPEC σ/Δλ ≈ {sig_bspec / dlam:.1f} pixels")

y_bspec = gaussian_rest(wave, lam0, sig_bspec)
n = wave.size
errs_a3 = []
errs_lin = []
for j in range(20, n - 21):
    xq = 0.5 * (wave[j] + wave[j + 1])
    ytrue = float(gaussian_rest(xq, lam0, sig_bspec))
    errs_a3.append(abs(_poly_interp_sorted_1d(xq, wave, y_bspec, n, 3) - ytrue))
    errs_lin.append(abs(_poly_interp_sorted_1d(xq, wave, y_bspec, n, 1) - ytrue))
errs_a3 = np.asarray(errs_a3)
errs_lin = np.asarray(errs_lin)
print(
    f"Lanczos-3 vs continuous at half-pixels: max|Δ|={errs_a3.max():.3e}  "
    f"median={np.median(errs_a3):.3e}"
)
print(
    f"linear vs continuous at half-pixels:    max|Δ|={errs_lin.max():.3e}  "
    f"median={np.median(errs_lin):.3e}"
)
assert errs_a3.max() < 1e-5, f"Lanczos-3 BSPEC interp max|Δ|={errs_a3.max()}"
assert errs_a3.max() < errs_lin.max(), "Lanczos-3 should beat linear on a resolved Gaussian"
print("PASS: Lanczos-3 on BSPEC samples recovers continuous flux far better than linear")

# Dense-rest Doppler paint with the same kernel (notebook / sim paint path).
rv_paint = 10_000.0
print(f"\n{'n_rest':>8}  {'max|Δ| paint':>14}")
last_err = None
for n_rest in (501, 2001, 8001, 32001, 128001):
    w_rest = np.linspace(lam0 - 20 * sig_bspec, lam0 + 20 * sig_bspec, n_rest)
    f_rest = gaussian_rest(w_rest, lam0, sig_bspec)
    f_paint = doppler_paint_lanczos(
        wave, w_rest, f_rest, rv_paint, order=MASK_CCF_PIXEL_INTERP_ORDER
    )
    beta = 1.0 / (1.0 + rv_paint / float(C_M_S))
    f_true = gaussian_rest(wave * beta, lam0, sig_bspec)
    m = (wave * beta > w_rest[5]) & (wave * beta < w_rest[-6])
    err = float(np.max(np.abs(f_paint[m] - f_true[m])))
    last_err = err
    print(f"{n_rest:8d}  {err:14.3e}")
print(f"PASS: densified Lanczos paint converges (final max|Δ|={last_err:.3e})")


BSPEC σ/Δλ ≈ 31.5 pixels
Lanczos-3 vs continuous at half-pixels: max|Δ|=2.059e-07  median=1.563e-225
linear vs continuous at half-pixels:    max|Δ|=1.258e-04  median=4.312e-225
PASS: Lanczos-3 on BSPEC samples recovers continuous flux far better than linear

  n_rest    max|Δ| paint
     501       9.430e-04
    2001       2.409e-04
    8001       6.054e-05
   32001       1.505e-05
  128001       3.729e-06
PASS: densified Lanczos paint converges (final max|Δ|=3.729e-06)


## 2. BT-Settl: densify + same Numba Lanczos paint (convergence)

Native BT-Settl is sorted, densified with `_upsample_spectrum`, then painted
with the same Lanczos-a kernel as the mask CCF.


In [4]:
from hispecdrp.primitives.utils.spectral_utils import (
    get_spectrum_path,
    load_btsettl_spectrum,
)
from hispecdrp.simulations.e2e_support import (
    DEFAULT_TEMPLATE_FILENAME,
    templates_data_dir,
)
from hispecdrp.simulations.simulate_rate_map import _upsample_spectrum

tpl_path = get_spectrum_path(DEFAULT_TEMPLATE_FILENAME, base_dir=templates_data_dir())
w_q, f_q = load_btsettl_spectrum(tpl_path)
w_native = np.asarray(w_q.value if hasattr(w_q, "value") else w_q, dtype=np.float64).ravel()
f_native = np.asarray(f_q.value if hasattr(f_q, "value") else f_q, dtype=np.float64).ravel()
order = np.argsort(w_native)
w_native, f_native = w_native[order], f_native[order]


def densify_btsettl(desired_R: float):
    wd, fd = _upsample_spectrum(
        w_native * w_q.unit,
        f_native * f_q.unit,
        desired_sampling_resolution=float(desired_R),
    )
    wr = np.asarray(wd.value if hasattr(wd, "value") else wd, dtype=np.float64).ravel()
    fr = np.asarray(fd.value if hasattr(fd, "value") else fd, dtype=np.float64).ravel()
    return wr, fr


w_hi, f_hi = densify_btsettl(3e6)
paint_hi = doppler_paint_lanczos(
    wave, w_hi, f_hi, rv_paint, order=MASK_CCF_PIXEL_INTERP_ORDER
)
beta = 1.0 / (1.0 + rv_paint / float(C_M_S))
m = (wave * beta > w_hi[10]) & (wave * beta < w_hi[-11])

print(f"{'R_densify':>10}  {'n_rest':>10}  {'max|Δ| vs 3e6':>14}")
for R in (3e5, 1e6, 3e6):
    wr, fr = densify_btsettl(R)
    p = doppler_paint_lanczos(
        wave, wr, fr, rv_paint, order=MASK_CCF_PIXEL_INTERP_ORDER
    )
    dabs = float(np.max(np.abs(p[m] - paint_hi[m])))
    print(f"{R:10.0e}  {wr.size:10d}  {dabs:14.3e}")

print("PASS: BT-Settl Lanczos paint converges under densification (same interpolator as CCF)")


 R_densify      n_rest   max|Δ| vs 3e6
     3e+05      635113       2.382e-18
     1e+06     1603382       1.505e-18
     3e+06     4245931       0.000e+00
PASS: BT-Settl Lanczos paint converges under densification (same interpolator as CCF)


## 3. One-line mask + analytical WAVE paint → Gaussian-centroid RV

Paint an exact Doppler-shifted Gaussian absorption onto WAVE (closed form).
Delta mask at rest `λ₀`. `StellarRVPrimitive` uses Numba Lanczos mask CCF
(`mask_ccf_pixel_interp_order=3`, `max_cores`) + **Gaussian centroid** only.

Use a line width of several BSPEC pixels so the discrete WAVE sampling does
not under-resolve the profile for the CCF Gaussian fit.


In [5]:
def analytical_doppler_gaussian(wave_obs, lam_c, sig, rv_m_s, zb=0.0):
    beta = (1.0 + float(zb)) / (1.0 + float(rv_m_s) / float(C_M_S))
    return gaussian_rest(wave_obs * beta, lam_c, sig)


from hispecdrp.primitives import StellarRVPrimitive

MASK_PATH = OUT / "single_gaussian_line.delta.mas"
MaskLines(
    mask_style="delta",
    line_center=np.array([lam0]) * u.um,
    line_left_edge=np.array([lam0]) * u.um,
    line_right_edge=np.array([lam0]) * u.um,
    line_width=np.array([0.0]) * u.um,
    weight=np.array([1.0]),
    metadata={"note": "single-line ideal-spectrum debug", "lam0_um": lam0},
).save(str(MASK_PATH))
print("mask:", MASK_PATH)


def build_l2_analytical_gaussian(rv_km_s: float) -> datamodels.L2Model:
    """Exact Doppler-shifted Gaussian absorption samples on instrument WAVE."""
    inj_m_s = float(rv_km_s) * 1000.0
    f_obs = 1.0 - 0.5 * analytical_doppler_gaussian(wave, lam0, sig_lam, inj_m_s)
    ost = oen = int(ORDER)
    model = datamodels.L2Model(
        meta={
            "order_start": ost,
            "order_end": oen,
            "spectrograph": SPECTROGRAPH,
            "instrument_name": "HISPEC",
        }
    )
    model.meta.order_start = ost
    model.meta.order_end = oen
    setattr(model, f"wave_fib{FIBER}", wave.reshape(1, -1).copy())
    setattr(model, f"spec_fib{FIBER}", f_obs.reshape(1, -1).astype(np.float32))
    setattr(
        model,
        f"spec_err_fib{FIBER}",
        np.full((1, wave.size), 1.0e-6, dtype=np.float32),
    )
    setattr(
        model,
        f"spec_dq_fib{FIBER}",
        np.zeros((1, wave.size), dtype=np.uint32),
    )
    return model


def recover_rv_m_s(rv_km_s: float) -> tuple[float, float]:
    inj = float(rv_km_s) * 1000.0
    out = StellarRVPrimitive.call(
        build_l2_analytical_gaussian(rv_km_s),
        algorithm="ccf_line_mask",
        mask_path=str(MASK_PATH),
        ccf_input_velocity=inj,
        do_fibers=[str(FIBER)],
        do_orders=[ORDER],
        window_line_by_line=True,
        window_size_um=float(max(5.65e-5, 10.0 * sig_lam)),
        v_half_window=float(V_HALF_WINDOW_M_S),
        v_step_coarse=float(V_STEP_M_S),
        mask_auto_center_scout=False,
        mask_ccf_pixel_interp_order=3,
        ccf_min_good_pixels=9,
        ccf_zb=0.0,
        ccf_zb_from_barycorr=False,
        save_result=False,
        save_preview=False,
        show_preview=False,
    )
    avg = getattr(out, f"rvccfavg_fib{FIBER}")
    return float(avg["RV"][0]), float(avg["RV_ERR"][0])


mask: test_data/ideal_spectrum_rv_recovery/single_gaussian_line.delta.mas


In [6]:
hdr = (
    f"{'bulk_km/s':>10}  {'injected_m/s':>14}  {'recovered_m/s':>16}  "
    f"{'bias_m/s':>12}  {'bias_mm/s':>10}"
)
print(hdr)
print("-" * len(hdr))

biases = []
for bulk in BULK_RV_KM_S:
    inj = bulk * 1000.0
    rec, err = recover_rv_m_s(bulk)
    bias = rec - inj
    biases.append(bias)
    print(
        f"{bulk:10.1f}  {inj:14.4f}  {rec:16.6f}  {bias:12.6e}  {bias * 1e3:10.4f}"
    )

biases = np.asarray(biases, dtype=float)
max_abs = float(np.max(np.abs(biases)))
print()
print(f"max|bias| = {max_abs:.6e} m/s ({max_abs * 1e3:.4f} mm/s)")
assert max_abs < RV_BIAS_MAX_M_S, (
    f"single-line |bias|={max_abs} m/s exceeds {RV_BIAS_MAX_M_S} m/s"
)
print(
    f"PASS: single-line Gaussian-centroid |bias| < {RV_BIAS_MAX_M_S:g} m/s "
    f"for all bulks (same one-line mask)"
)


 bulk_km/s    injected_m/s     recovered_m/s      bias_m/s   bias_mm/s
----------------------------------------------------------------------
[2026-08-22T00:26:14][hispecdrp.datamodels.utils:289][WARNING]: No time info is present, adopting current time for datetime_obs and mjd_start.
[2026-08-22T00:26:14][hispecdrp.primitives.base_primitive:282][INFO]: Applying StellarRVPrimitive with arguments:
input = L2Model()
run = True
save_result = False
output_model = L2Model
output_file = None
output_dir = None
suffix = stellar_rv
save_preview = False
show_preview = False
algorithm = ccf_line_mask
do_fibers = ['1']
do_orders = [130]
template_path = None
template_wavelength = None
template_flux = None
mask_path = test_data/ideal_spectrum_rv_recovery/single_gaussian_line.delta.mas
mask_wavelength = None
mask_weight = None
mask_ccf_pixel_interp_order = 3
mask_auto_center_scout = False
window_size_um = 0.00037469955063054993
outlier_sigma = 3.0
chunk_rv_max_abs_m_s = 120000.0
chunk_rv_prior_half_wi

AssertionError: single-line |bias|=1421.0985443894897 m/s exceeds 0.001 m/s